# ⏱️ EXP-010: Temporal Kinematic Fall Velocity & Dynamic Sequence Analysis

Notebook ini mengeksplorasi **Dimensi Waktu (Temporal Dynamics)** untuk membedakan antara:
1. **Jatuh Mendadak (Sudden Fall)**: Terjadi lonjakan kecepatan vertikal ($V_y = \frac{\Delta y}{\Delta t}$) yang tajam akibat gaya gravitasi bumi ($g = 9.8\text{ m/s}^2$).
2. **Aktivitas Normal Terkontrol (ADL)**: Duduk di kursi, membungkuk mengambil barang, atau rebahan santai di kasur yang memiliki profil kecepatan halus dan terkendali.

### 🎯 Tujuan Eksperimen:
- Menganalisis kurva trajektori sudut $\theta(t)$ dan kecepatan vertikal $V_y(t)$ per rangkaian video.
- Menemukan threshold kecepatan jatuh optimal pada Validation Set.
- Menguji **Dual-Trigger Fall Confirmation**: Orang dianggap jatuh jika dan hanya jika $(\theta < 25.0^\circ)$ **DAN** pernah mengalami lonjakan kecepatan jatuh ($V_{y,\max} \ge V_{\text{threshold}}$).

In [ ]:
import math
import time
from pathlib import Path
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

print("🚀 Memuat model YOLO11-Pose...")
model = YOLO("yolo11n-pose.pt")
print("✅ Model siap!")

In [ ]:
def calculate_spine_angle(kpts: np.ndarray) -> float:
    l_sh, r_sh = kpts[5][:2], kpts[6][:2]
    l_hip, r_hip = kpts[11][:2], kpts[12][:2]
    mid_sh = ((l_sh[0] + r_sh[0]) / 2, (l_sh[1] + r_sh[1]) / 2)
    mid_hip = ((l_hip[0] + r_hip[0]) / 2, (l_hip[1] + r_hip[1]) / 2)
    dx = mid_sh[0] - mid_hip[0]
    dy = mid_sh[1] - mid_hip[1]
    return math.degrees(math.atan2(abs(dy), abs(dx)))

def get_sequences(img_dir: Path):
    seqs = defaultdict(list)
    for f in img_dir.glob("*.jpg"):
        prefix = f.name.split("_")[0]
        try:
            f_num = int(f.name.split("_f")[1].split(".")[0])
        except (IndexError, ValueError):
            continue
        seqs[prefix].append((f_num, f))
    for s in seqs:
        seqs[s].sort(key=lambda x: x[0])
    return seqs

def extract_sequence_dynamics(seq_frames, model, k_window: int = 2):
    angles, hip_y_norm = [], []
    for frame_idx, (f_num, img_path) in enumerate(seq_frames):
        res = model(str(img_path), verbose=False, conf=0.25)
        if not res or len(res[0].boxes) == 0:
            angles.append(90.0 if not angles else angles[-1])
            hip_y_norm.append(0.5 if not hip_y_norm else hip_y_norm[-1])
            continue
            
        boxes = res[0].boxes
        best_idx = int(boxes.conf.argmax().item())
        kpts = res[0].keypoints.data[best_idx].cpu().numpy()
        bbox = boxes.xyxy[best_idx].tolist()
        h_box = max(1.0, bbox[3] - bbox[1])
        img_h = res[0].orig_shape[0]
        
        sh_conf = (kpts[5][2] + kpts[6][2]) / 2
        hip_conf = (kpts[11][2] + kpts[12][2]) / 2
        
        if sh_conf > 0.25 and hip_conf > 0.25:
            ang = calculate_spine_angle(kpts)
            mid_hip_y = (kpts[11][1] + kpts[12][1]) / 2.0
        else:
            ar = (bbox[2] - bbox[0]) / h_box
            ang = 90.0 if ar < 0.8 else (20.0 if ar > 1.3 else 50.0)
            mid_hip_y = (bbox[1] + bbox[3]) / 2.0
            
        angles.append(ang)
        hip_y_norm.append(mid_hip_y / img_h)
        
    angles = np.array(angles)
    hip_y_norm = np.array(hip_y_norm)
    velocities = np.zeros_like(hip_y_norm)
    for t in range(k_window, len(hip_y_norm)):
        velocities[t] = (hip_y_norm[t] - hip_y_norm[t - k_window]) / k_window
        
    return {
        "angles": angles,
        "hip_y": hip_y_norm,
        "velocities": velocities,
        "min_angle": float(np.min(angles)),
        "max_velocity": float(np.max(velocities)),
    }

In [ ]:
dataset_dir = Path("../data/processed") if Path("../data/processed").exists() else Path("ml/data/processed")
val_img_dir = dataset_dir / "images" / "val"
test_img_dir = dataset_dir / "images" / "test"

print("⏳ Mengekstrak dinamika pada Validation Set...")
val_seqs = get_sequences(val_img_dir)
fall_vels_val, adl_vels_val = [], []

for s_name, s_frames in sorted(val_seqs.items()):
    dyn = extract_sequence_dynamics(s_frames, model)
    if s_name.startswith("fall"):
        fall_vels_val.append(dyn["max_velocity"])
    else:
        adl_vels_val.append(dyn["max_velocity"])
    print(f"  • {s_name}: Min Angle={dyn['min_angle']:.1f}°, Max Downward Vel={dyn['max_velocity']:.4f}")

vel_threshold = min((np.mean(fall_vels_val) + np.mean(adl_vels_val)) / 2.0, 0.035)
print(f"\n🎯 Optimal Fall Velocity Threshold: Vy > {vel_threshold:.4f}")

In [ ]:
print("⏳ Mengevaluasi pada Test Set...")
test_seqs = get_sequences(test_img_dir)
test_results = {}
y_true, y_pred_static, y_pred_temporal = [], [], []

for s_name, s_frames in sorted(test_seqs.items()):
    dyn = extract_sequence_dynamics(s_frames, model)
    test_results[s_name] = dyn
    gt = 1 if s_name.startswith("fall") else 0
    y_true.append(gt)
    
    pred_stat = 1 if dyn["min_angle"] < 25.0 else 0
    y_pred_static.append(pred_stat)
    
    pred_temp = 1 if (dyn["min_angle"] < 25.0 and dyn["max_velocity"] >= vel_threshold) else 0
    y_pred_temporal.append(pred_temp)
    verdict = "✅ BENAR" if pred_temp == gt else "❌ SALAH"
    print(f"  • {s_name:<8} [GT: {'FALL' if gt else 'ADL '}] -> Pred: {'FALL' if pred_temp else 'ADL '} | MinAngle: {dyn['min_angle']:4.1f}° | MaxVel: {dyn['max_velocity']:.4f} | {verdict}")

def print_metrics(name, y_t, y_p):
    tp = sum(1 for t, p in zip(y_t, y_p) if t == 1 and p == 1)
    fp = sum(1 for t, p in zip(y_t, y_p) if t == 0 and p == 1)
    fn = sum(1 for t, p in zip(y_t, y_p) if t == 1 and p == 0)
    tn = sum(1 for t, p in zip(y_t, y_p) if t == 0 and p == 0)
    acc = (tp + tn) / len(y_t)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    print(f"{name:<25}: Accuracy={acc*100:.1f}%, Precision={prec*100:.1f}%, Recall={rec*100:.1f}%, F1={f1*100:.1f}%, FP={fp}")

print("\n📊 PERBANDINGAN METRIK PADA TEST SET:")
print_metrics("Static Angle Only", y_true, y_pred_static)
print_metrics("Temporal Velocity (EXP-010)", y_true, y_pred_temporal)

In [ ]:
sample_fall = test_results.get("fall-26", list(test_results.values())[0])
sample_adl = test_results.get("adl-35", list(test_results.values())[-1])

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes[0, 0].plot(sample_fall["angles"], color="red", lw=2, label="Spine Angle θ (°)")
axes[0, 0].axhline(20.0, color="darkred", linestyle="--", label="Lying Threshold (20°)")
axes[0, 0].set_title("Fall Sequence: Profil Sudut Tulang Belakang", fontsize=11, weight="bold")
axes[0, 0].set_ylabel("Sudut (Derajat)")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[1, 0].plot(sample_fall["velocities"], color="crimson", lw=2, label="Downward Velocity Vy")
axes[1, 0].axhline(vel_threshold, color="blue", linestyle="--", label=f"Threshold ({vel_threshold:.3f})")
axes[1, 0].set_title("Fall Sequence: Kecepatan Jatuh Vertikal (Spike!)", fontsize=11, weight="bold")
axes[1, 0].set_xlabel("Frame Index")
axes[1, 0].set_ylabel("Velocity (norm_h / frame)")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

axes[0, 1].plot(sample_adl["angles"], color="green", lw=2, label="Spine Angle θ (°)")
axes[0, 1].axhline(20.0, color="darkred", linestyle="--", label="Lying Threshold (20°)")
axes[0, 1].set_title("ADL Sequence: Profil Sudut Normal", fontsize=11, weight="bold")
axes[0, 1].set_ylabel("Sudut (Derajat)")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

axes[1, 1].plot(sample_adl["velocities"], color="seagreen", lw=2, label="Downward Velocity Vy")
axes[1, 1].axhline(vel_threshold, color="blue", linestyle="--", label=f"Threshold ({vel_threshold:.3f})")
axes[1, 1].set_title("ADL Sequence: Kecepatan Lembut & Terkontrol", fontsize=11, weight="bold")
axes[1, 1].set_xlabel("Frame Index")
axes[1, 1].set_ylabel("Velocity (norm_h / frame)")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()